In [1]:
import numpy as np
import json
import pandas as pd


In [2]:
TRAJ_PATH = "../logs/stage1_trajectories_1783860850.json"
with open(TRAJ_PATH, "r") as f:
    trajs = json.load(f)

print("Number of turns in the dataset:", len(trajs))

Number of turns in the dataset: 469


In [3]:
# getting the fields of the datasets
fields = list(trajs[0].keys())
fields

['scenario_index',
 'turn_index',
 'state',
 'incoming_message',
 'thought',
 'doctor_action',
 'raw_model_output',
 'parsed_ok']

In [4]:
from enum import Enum

class TrajectoryField(str, Enum):
    SCENARIO_INDEX = "scenario_index"
    TURN_INDEX = "turn_index"
    STATE = "state"
    INCOMING_MESSAGE = "incoming_message"
    THOUGHT = "thought"
    DOCTOR_ACTION = "doctor_action"
    RAW_MODEL_OUTPUT = "raw_model_output"
    PARSED_OK = "parsed_ok"

In [5]:
# Creating a DataFrame from the list of dictionaries
df = pd.DataFrame(trajs)

n = len(df)
print(f"Total turns logged: {n}")

parse_rate = df[TrajectoryField.PARSED_OK].mean()
print(f"Parse success rate: {100 * parse_rate:.1f}%  ({df[TrajectoryField.PARSED_OK].sum()}/{n})")

empty_thoughts = df[TrajectoryField.THOUGHT].fillna("").str.strip().eq("").sum()
print(f"Empty thoughts: {empty_thoughts}  ({100 * empty_thoughts / n:.1f}%)")

empty_actions = df[TrajectoryField.DOCTOR_ACTION].fillna("").str.strip().eq("").sum()
print(f"Empty actions: {empty_actions}  ({100 * empty_actions / n:.1f}%)")

mismatch = df[(~df[TrajectoryField.PARSED_OK]) & (df[TrajectoryField.THOUGHT].fillna("").str.strip() != "")]
print(f"parsed_ok=False but thought non-empty (unexpected): {len(mismatch)}")

parsed_ok_empty = df[(df[TrajectoryField.PARSED_OK]) & (df[TrajectoryField.THOUGHT].fillna("").str.strip() == "")]
print(f"parsed_ok=True but thought empty (unexpected): {len(parsed_ok_empty)}")



Total turns logged: 469
Parse success rate: 97.9%  (459/469)
Empty thoughts: 10  (2.1%)
Empty actions: 0  (0.0%)
parsed_ok=False but thought non-empty (unexpected): 0
parsed_ok=True but thought empty (unexpected): 0


In [6]:
turns_per_scenario = df.groupby(TrajectoryField.SCENARIO_INDEX).size()

print(f"Number of scenarios: {df[TrajectoryField.SCENARIO_INDEX].nunique()}")
print(f"Total turns: {len(df)}")
print()
print("Turns per scenario:")
print(turns_per_scenario.describe())
print()
print(f"Scenarios with <= 3 turns: {(turns_per_scenario <= 3).sum()}")
print()
print("Full distribution (scenario_index -> turn count):")
print(turns_per_scenario.sort_values())

Number of scenarios: 30
Total turns: 469

Turns per scenario:
count    30.000000
mean     15.633333
std       4.189341
min       7.000000
25%      12.250000
50%      16.500000
75%      20.000000
max      20.000000
dtype: float64

Scenarios with <= 3 turns: 0

Full distribution (scenario_index -> turn count):
scenario_index
13     7
0      9
1      9
3     10
17    10
7     11
12    11
11    12
4     13
27    14
19    14
5     15
28    15
15    15
10    16
23    17
14    17
6     17
18    18
26    19
9     20
8     20
20    20
21    20
22    20
24    20
25    20
2     20
16    20
29    20
dtype: int64


In [7]:
import json

with open("../logs/stage1_baseline_1783860850.json") as f:
    results = json.load(f)

diagnosed_count = len(results)
print(f"Scenarios that reached a diagnosis: {diagnosed_count} / 30")
print(f"Scenarios that hit turn budget without diagnosing: {30 - diagnosed_count}")

Scenarios that reached a diagnosis: 30 / 30
Scenarios that hit turn budget without diagnosing: 0


In [ ]:
data  = pd.read_json("../logs/stage1_embedded_1784192669.jsonl", lines=True)


In [15]:
len(data['state_embedding'][1])

384